# Mini Model UN · Global AI & Climate Adaptation Fund (all-in-one classroom notebook)

This repository’s **single classroom entry** is this notebook. Simulation and the web UI still run through the `scripts/` modules (same code as running `python` from the terminal).

---

## Before you start

1. **Git** and **Python 3.10+** (on Windows, enable *Add Python to PATH*).
2. **Clone** the repo and open this notebook from the project root in Jupyter / VS Code / Cursor, or use the first code cell to auto-detect root.
3. **Configure `.env`**: copy `.env.example` → `.env` (**DeepSeek is the default**). At minimum set:
   - `LLM_API_KEY` — [DeepSeek console](https://platform.deepseek.com/api_keys)
   - `LLM_BASE_URL` — default `https://api.deepseek.com` (do **not** append `/v1`)
   - `LLM_MODEL_NAME` — e.g. `deepseek-chat` or `deepseek-v4-flash` (match your dashboard)
   - **OpenAI** only if your instructor allows: `https://api.openai.com/v1` and the matching model id
4. **Cost**: baseline makes many model calls; the browser **Continue discussion** feature charges per selected delegate.

---

## Recommended cell order

| Step | Cell | Purpose |
|------|------|--------|
| 1 | **Project root** | `cd` to the folder that contains `scripts/run_minisim.py` |
| 2 | **Dependencies** | `pip install` requirements |
| 3 | **Preflight** | Python version, `.env`, required vars (key never printed) |
| 4 | **Optional checks** | Smoke tests + short provider probe + optional cost estimate |
| 5 | **Run simulation** | Writes `outputs/model_un_*` files |
| 6 | **Start lab UI** | Opens `http://127.0.0.1:8080/` (change port if needed) |
| 7 | **Stop lab UI** | Free the port |

If **8080 is busy**, set `LAB_UI_PORT = 8090` (or another free port) in the **Start lab UI** cell.

---

## Docs & troubleshooting

- **Full student guide:** `docs/STUDENT_LAB_GUIDE.md`
- **Short student README:** `README_STUDENTS.md`
- **Ports / keys / models:** `docs/troubleshooting.md`
- Outputs are **scenario deliberation** for class reflection, not factual prediction.
- You may omit `ZEP_API_KEY` (reduced-memory classroom mode).


In [ ]:
"""Locate project root and change working directory (notebook may live in root or a subfolder)."""
from __future__ import annotations

import os
from pathlib import Path


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "scripts" / "run_minisim.py").is_file():
            return d
    raise FileNotFoundError(
        "Could not find project root (need scripts/run_minisim.py).\n"
        "Open Terminal / Jupyter from the repo folder, or File → Open Folder on the repo root, then open this notebook."
    )


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)


In [ ]:
# Install dependencies (safe to re-run inside a venv)
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")]
)
# Local Jupyter Lab/Classic often needs ipykernel (small package)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipykernel"])
print("Dependencies installed (requirements.txt + ipykernel).")


In [ ]:
"""Preflight: Python version, .env, required vars (never prints the secret key)."""
import os
import sys
from pathlib import Path

if sys.version_info < (3, 10):
    raise RuntimeError("Python 3.10+ required. Current: " + sys.version)

from dotenv import load_dotenv

env_path = PROJECT_ROOT / ".env"
if not env_path.is_file():
    raise FileNotFoundError(
        f"Missing {env_path}\nCopy template: cp .env.example .env  (Windows: copy .env.example .env)"
    )

load_dotenv(env_path)

key = (os.getenv("LLM_API_KEY") or "").strip()
base = (os.getenv("LLM_BASE_URL") or "").strip()
model = (os.getenv("LLM_MODEL_NAME") or "").strip()

errs = []
if not key or key.startswith("PASTE_"):
    errs.append("LLM_API_KEY missing or still a placeholder")
if not base:
    errs.append("LLM_BASE_URL is empty")
if not model:
    errs.append("LLM_MODEL_NAME is empty")

if errs:
    raise RuntimeError("; ".join(errs) + "\nEdit .env and re-run this cell.")

print("Preflight OK — Python", sys.version.split()[0])
print("LLM_BASE_URL =", base)
print("LLM_MODEL_NAME =", model)
print("LLM_API_KEY is set (length", len(key), "chars; value hidden)")


In [ ]:
# Optional: classroom smoke (formerly smoke_test_classroom) + API probe (formerly test_provider) + cost estimate (formerly cost_guard)
# All inlined here; does not depend on removed standalone .py launchers.
from __future__ import annotations

import io
import json
import os
import py_compile
import re
import sys
from contextlib import redirect_stdout
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

RUN_OPTIONAL_TESTS = True  # False skips smoke + test_provider
RUN_COST_ESTIMATE = True  # Approximate counts only; no API cost for this block

if not RUN_OPTIONAL_TESTS:
    print("Skipped optional checks (RUN_OPTIONAL_TESTS=False).")
else:
    load_dotenv(PROJECT_ROOT / ".env")

    REQUIRED_ENV = [
        "LLM_API_KEY",
        "LLM_BASE_URL",
        "LLM_MODEL_NAME",
        "MAX_AGENTS",
        "MAX_ROUNDS",
        "MAX_SEED_WORDS",
        "MAX_REPORT_WORDS",
    ]
    CASE_FILES = [
        "case_materials/seed_un_ai_climate_governance.txt",
        "case_materials/prediction_request.txt",
        "case_materials/agent_roles.md",
    ]

    print("[smoke 1/6] Required env vars …")
    miss = [k for k in REQUIRED_ENV if not os.getenv(k)]
    if miss:
        raise RuntimeError("Missing: " + ", ".join(miss))

    print("[smoke 2/6] Classroom caps …")
    limits = {
        "MAX_AGENTS": int(os.getenv("MAX_AGENTS", "0")),
        "MAX_ROUNDS": int(os.getenv("MAX_ROUNDS", "0")),
        "MAX_SEED_WORDS": int(os.getenv("MAX_SEED_WORDS", "0")),
        "MAX_REPORT_WORDS": int(os.getenv("MAX_REPORT_WORDS", "0")),
    }
    expected = {
        "MAX_AGENTS": 5,
        "MAX_ROUNDS": 3,
        "MAX_SEED_WORDS": 1000,
        "MAX_REPORT_WORDS": 800,
    }
    for k, vmax in expected.items():
        if limits[k] > vmax:
            raise RuntimeError(f"{k} exceeds classroom cap: {limits[k]} > {vmax}")

    print("[smoke 3/6] Case files …")
    for rel in CASE_FILES:
        if not (PROJECT_ROOT / rel).is_file():
            raise RuntimeError(f"Missing {rel}")

    print("[smoke 4/6] outputs/ writable …")
    out = PROJECT_ROOT / "outputs"
    out.mkdir(exist_ok=True)
    probe = out / ".smoke_probe"
    probe.write_text("ok", encoding="utf-8")
    probe.unlink(missing_ok=True)

    print("[smoke 5/6] run_minisim.py syntax …")
    mini = PROJECT_ROOT / "scripts" / "run_minisim.py"
    if not mini.is_file():
        raise RuntimeError("Missing scripts/run_minisim.py")
    py_compile.compile(str(mini), doraise=True)

    if not os.getenv("ZEP_API_KEY") or os.getenv("ZEP_API_KEY", "").startswith("PASTE_"):
        print("WARNING: ZEP_API_KEY not set → reduced-memory classroom mode")

    print("[smoke 6/6] No secret leakage to stdout …")
    key = os.getenv("LLM_API_KEY", "")
    buf = io.StringIO()
    with redirect_stdout(buf):
        print("smoke_probe_output")
    leaked = buf.getvalue()
    if key and key in leaked:
        raise RuntimeError("Smoke check bug: do not print secrets to stdout")
    if re.search(r"sk-[A-Za-z0-9]", leaked):
        raise RuntimeError("Smoke: possible API-key-shaped text leaked to output")

    print("PASS: classroom smoke complete.")

    def _parse_json_from_text(text: str):
        text = text.strip()
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            start, end = text.find("{"), text.rfind("}")
            if start != -1 and end > start:
                return json.loads(text[start : end + 1])
            raise

    print("\n[provider] Sending minimal API request (same idea as old test_provider) …")
    pkey = (os.getenv("LLM_API_KEY") or "").strip()
    base_url = (os.getenv("LLM_BASE_URL") or "").strip()
    model = (os.getenv("LLM_MODEL_NAME") or "").strip()
    miss2 = [
        n
        for n, v in [
            ("LLM_API_KEY", pkey),
            ("LLM_BASE_URL", base_url),
            ("LLM_MODEL_NAME", model),
        ]
        if not v or v.startswith("PASTE_")
    ]
    if miss2:
        raise RuntimeError("provider: " + ", ".join(miss2))

    client = OpenAI(api_key=pkey, base_url=base_url)
    messages = [
        {"role": "system", "content": "Return JSON only."},
        {"role": "user", "content": "Return an object with key 'status' and value 'ok'."},
    ]
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0,
            response_format={"type": "json_object"},
        )
        content = resp.choices[0].message.content or "{}"
        data = json.loads(content)
        print(f"SUCCESS: JSON mode works. status={data.get('status')}")
    except Exception as e1:
        print(f"WARNING: JSON mode failed ({type(e1).__name__}). Fallback …")
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages, temperature=0
            )
            content = resp.choices[0].message.content or ""
            data = _parse_json_from_text(content)
            print(f"SUCCESS: Fallback parsing works. status={data.get('status')}")
        except Exception as e2:
            outf = PROJECT_ROOT / "outputs" / "provider_test_raw_response.txt"
            outf.write_text(
                f"Provider test failed.\n{type(e2).__name__}: {e2}\n", encoding="utf-8"
            )
            raise RuntimeError(
                f"Provider test failed; see {outf}. Check network, model name, and key."
            ) from e2

    print("\nAll optional checks passed.")

if RUN_COST_ESTIMATE:
    agents, rounds = int(os.getenv("MAX_AGENTS", "5")), int(os.getenv("MAX_ROUNDS", "3"))
    agent_calls = agents * rounds
    summary_calls = rounds
    report_calls = 1
    total = agent_calls + summary_calls + report_calls
    print("\n--- Cost estimate (former cost_guard logic) ---")
    print(f"{agents} agents × {rounds} rounds = {agent_calls} agent calls")
    print(f"+ {summary_calls} round summaries + {report_calls} final report ≈ {total} API calls (baseline)")
    print("Actual billing is token-based; 'Continue discussion' charges per checked delegate.\n")


## Run baseline simulation

The next code cell runs `scripts/run_minisim.py` (same as the CLI) and writes the transcript and report under `outputs/`.

**Re-running overwrites** baseline files with the same names (continuations in `model_un_student_continuations.json` are not deleted by this script, but a new baseline transcript becomes the reference for the UI).


In [ ]:
"""Run Mini Model UN baseline (many API calls; may take minutes depending on network and model)."""
import os
import subprocess
import sys

print("Starting run_minisim.py …")
proc = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "scripts" / "run_minisim.py")],
    cwd=str(PROJECT_ROOT),
    env=os.environ.copy(),
)
if proc.returncode != 0:
    raise RuntimeError(
        f"Simulation failed (exit {proc.returncode}). Check output above: billing, model id, proxy/firewall."
    )

out = PROJECT_ROOT / "outputs"
expected = [
    out / "model_un_transcript.json",
    out / "model_un_simulation_report.md",
]
for p in expected:
    print("Wrote:", p.name, "OK" if p.is_file() else "MISSING")

print("\nDone. Next: start the lab UI in the browser.")


## Browser lab UI (view results + **Continue discussion**)

- The **Start lab UI** cell launches `scripts/serve_lab_ui.py` in the **background** and tries to open your default browser.
- **Do not start twice** on the same port without stopping the old process first.
- Change **`LAB_UI_PORT`** if 8080 is taken (e.g. 8090).


In [ ]:
"""Start lab web UI (background); open default browser."""
import subprocess
import sys
import time
import webbrowser

LAB_UI_HOST = "127.0.0.1"
LAB_UI_PORT = 8080  # change to 8090 if busy

script = PROJECT_ROOT / "scripts" / "serve_lab_ui.py"
url = f"http://{LAB_UI_HOST}:{LAB_UI_PORT}/"

_lab_ui_proc = globals().get("_lab_ui_proc")
if _lab_ui_proc is not None and _lab_ui_proc.poll() is None:
    print("A lab UI process is already running (PID", _lab_ui_proc.pid, "). Run the **Stop lab UI** cell first.")
else:
    _lab_ui_proc = subprocess.Popen(
        [sys.executable, str(script), "--host", LAB_UI_HOST, "--port", str(LAB_UI_PORT)],
        cwd=str(PROJECT_ROOT),
    )
    globals()["_lab_ui_proc"] = _lab_ui_proc
    time.sleep(0.8)
    webbrowser.open(url)
    print("Lab UI:", url)
    print("PID:", _lab_ui_proc.pid, "— use **Stop lab UI** below to shut down.")


In [ ]:
"""Stop the lab UI background process."""
import subprocess

p = globals().get("_lab_ui_proc")
if p is None or p.poll() is not None:
    print("No lab UI process running (or already exited).")
else:
    p.terminate()
    try:
        p.wait(timeout=5)
    except subprocess.TimeoutExpired:
        p.kill()
        p.wait(timeout=3)
    print("Lab UI stopped.")
